# Whisper realtime server on a free GPU (Kaggle / Colab)Runs this repo's WhisperLiveKit server on a cloud GPU and exposes it through anHTTPS tunnel so your browser mic can reach it.**Before you run anything**, in the right-hand sidebar set:- **Accelerator** -> `GPU T4 x2`- **Internet** -> `On`  (needs a phone-verified Kaggle account)Then run the cells in order. The last cell prints a `https://...trycloudflare.com`link - open that, allow the microphone, and press record.Keep this browser tab open. When the session ends, the server dies with it.

## 1. Install dependencies

In [ ]:
!pip install -q faster-whisper whisperlivekit nvidia-cublas-cu12 nvidia-cudnn-cu12print("installed")

## 2. Make the CUDA libraries visible`ctranslate2` (the engine behind faster-whisper) loads cuBLAS and cuDNN at runtime.The pip wheels put them somewhere the loader does not look by default, so add themto `LD_LIBRARY_PATH` before the server starts.

In [ ]:
import os, sitelibs = []for sp in site.getsitepackages():    nv = os.path.join(sp, "nvidia")    if os.path.isdir(nv):        for sub in sorted(os.listdir(nv)):            d = os.path.join(nv, sub, "lib")            if os.path.isdir(d):                libs.append(d)os.environ["LD_LIBRARY_PATH"] = ":".join(libs + [os.environ.get("LD_LIBRARY_PATH", "")])print(f"{len(libs)} CUDA lib dirs added")import ctranslate2print("CUDA devices visible to ctranslate2:", ctranslate2.get_cuda_device_count())

## 3. Get the code and apply the browser mic fix`patch_webui.py` is required. Without it the page loads but no audio ever reachesthe server ("No audio detected..."). See the README for what it fixes.

In [ ]:
import os, subprocessREPO = "https://github.com/kunalost/whisper.git"DIR = "/kaggle/working/whisper" if os.path.isdir("/kaggle/working") else "/content/whisper"if not os.path.isdir(DIR):    subprocess.run(["git", "clone", "--depth", "1", REPO, DIR], check=True)else:    subprocess.run(["git", "-C", DIR, "pull"], check=True)subprocess.run(["python", "patch_webui.py"], cwd=DIR, check=True)print("repo at", DIR)

## 4. Install cloudflared (the tunnel)Kaggle and Colab give you no public inbound port, so the browser cannot reach`localhost:8000` directly. cloudflared opens an outbound connection and hands backa public **https** URL. HTTPS matters: browsers block microphone access on plainhttp, so an http tunnel would load the page but never capture audio.

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared!chmod +x /usr/local/bin/cloudflared!cloudflared --version

## 5. Start the serverFirst run downloads the model: `large-v3` is about 3 GB, so give it a few minutes.Use `--model large-v3-turbo` if you want it faster, or `--language hi` to force Hindi(auto-detect often picks the wrong language on short live audio).

In [ ]:
import subprocess, time, urllib.request, osLOG = "/tmp/wlk.log"args = [    "python", "run_wlk.py",    "--model", "large-v3",    "--language", "auto",    "--backend", "faster-whisper",    "--backend-policy", "localagreement",    "--port", "8000",    "--pcm-input",    "--min-chunk-size", "0.5",]env = dict(os.environ, WLK_DEVICE="cuda", WLK_COMPUTE="float16")log = open(LOG, "w")server = subprocess.Popen(args, cwd=DIR, env=env, stdout=log, stderr=subprocess.STDOUT)print("server pid", server.pid, "- waiting for model to load...")for i in range(90):    if server.poll() is not None:        print("SERVER EXITED. Last log lines:")        print(open(LOG).read()[-3000:])        break    try:        with urllib.request.urlopen("http://localhost:8000/health", timeout=2) as r:            print("READY:", r.read().decode())            break    except Exception:        time.sleep(10)else:    print("timed out; log tail:")    print(open(LOG).read()[-3000:])

## 6. Open the tunnelThis cell keeps running - that is normal, it is holding the tunnel open.Look for the line containing `trycloudflare.com` and open that **https** URLin a new tab, then allow the microphone and press record.Stop this cell to close the tunnel.

In [ ]:
!cloudflared tunnel --url http://localhost:8000 --no-autoupdate

## Troubleshooting- **"No audio detected..."** - cell 3 did not run, or the repo copy is missing  `patch_webui.py`. Re-run cell 3, then hard-refresh the tunnel page (Ctrl+Shift+R).- **Mic permission never prompts** - you opened the `http://` URL. Use `https://`.- **`CUDA devices visible: 0`** in cell 2 - the Accelerator is not set to GPU.  Set it in the sidebar, then restart the session and run the cells again.- **Server exited in cell 5** - read the log tail it prints. A cuDNN/cuBLAS  "cannot open shared object file" error means cell 2 did not run before cell 5.- **Everything dies suddenly** - the session hit its time or idle limit. This is  expected on free tiers; restart and re-run.